# HRRR Storm Wind Data — Hartford County

Downloads **10-m wind speed**, **2-m temperature**, and **0–10 cm soil moisture**
from the NOAA HRRR model (AWS archive, `herbie-data`) for five Hartford County
storms. Outputs `hartford_storm_wind.js` to copy into `data/` of the simulation.

**Pre-HRRR storms** (Sandy 2012, Irene 2011, Snowtober 2011) need ERA5 — not covered here.

**Key technical note:** HRRR uses Lambert Conformal projection, so latitude/longitude
are 2-D arrays and `xr.interp()` won’t work. We clip to a Hartford County bounding box
and use `scipy.interpolate.griddata` to regrid scattered HRRR points onto a regular
lat/lon grid.

In [ ]:
!pip install herbie-data scipy -q

In [ ]:
import json
import warnings
import numpy as np
from scipy.interpolate import griddata
warnings.filterwarnings('ignore')
from herbie import Herbie
print('herbie-data loaded OK')

In [ ]:
# Hartford County target grid (matches data/hartford_storm_wind.js)
TARGET_LATS = np.array([
    41.35, 41.40, 41.45, 41.50, 41.55, 41.60, 41.65, 41.70,
    41.75, 41.80, 41.85, 41.90, 41.95, 42.00, 42.10
])
TARGET_LONS = np.array([
    -73.10, -73.04, -72.98, -72.92, -72.86, -72.80, -72.74,
    -72.68, -72.62, -72.56, -72.50, -72.44, -72.38, -72.32,
    -72.26, -72.20, -72.14, -72.08, -72.02, -71.96, -71.95
])
LON_GRID, LAT_GRID = np.meshgrid(TARGET_LONS, TARGET_LATS)

# HRRR extraction bounding box (Hartford County + 50-mile buffer)
LAT_MIN, LAT_MAX = 41.0, 42.5
LON_MIN, LON_MAX = -73.5, -71.5

# Peak-impact hour for each storm (UTC). Adjust if you want a different snapshot.
STORMS = {
    'isaias_2020': {'date': '2020-08-04 18:00', 'precip_type': 'rain'},
    'henri_2021':  {'date': '2021-08-22 14:00', 'precip_type': 'rain'},
    'may2018':     {'date': '2018-05-15 21:00', 'precip_type': 'rain'},
    'jan2024':     {'date': '2024-01-10 19:00', 'precip_type': 'mix'},
    'dec2023':     {'date': '2023-12-18 18:00', 'precip_type': 'snow'},
}
STORM_NAMES = {
    'isaias_2020': 'Tropical Storm Isaias',
    'henri_2021':  'Tropical Storm Henri',
    'may2018':     'May 2018 Tornadoes / Derecho',
    'jan2024':     'January 2024 Wind Storm',
    'dec2023':     "December 2023 Nor'easter",
}
n_pts = len(TARGET_LATS) * len(TARGET_LONS)
print(f'Grid: {len(TARGET_LATS)} lat x {len(TARGET_LONS)} lon = {n_pts} points')
print(f'Storms: {list(STORMS.keys())}')

In [ ]:
# HRRR uses Lambert Conformal projection.
# latitude/longitude are 2-D coordinate arrays (not dimension coords),
# so we can't use xr.interp(). Strategy:
#   1. Clip HRRR grid to a bounding box around Hartford County.
#   2. Use scipy griddata on the scattered (lon, lat) -> value points.
#   3. Interpolate onto our regular target grid.

def _first_var(ds):
    skip = {
        'latitude', 'longitude', 'valid_time', 'step', 'time',
        'surface', 'heightAboveGround', 'depthBelowLandLayer', 'level',
    }
    return next(v for v in ds.data_vars if v not in skip)

def extract_to_grid(ds):
    lats_2d = ds.latitude.values           # shape ~(1059, 1799) for CONUS HRRR
    lons_2d = ds.longitude.values          # 0-360 convention in HRRR grib2
    lons_neg = np.where(lons_2d > 180, lons_2d - 360, lons_2d)  # -> -180-180
    mask = (
        (lats_2d  >= LAT_MIN) & (lats_2d  <= LAT_MAX) &
        (lons_neg >= LON_MIN) & (lons_neg <= LON_MAX)
    )
    return griddata(
        (lons_neg[mask], lats_2d[mask]),
        ds[_first_var(ds)].values[mask],
        (LON_GRID, LAT_GRID),
        method='linear',
    )

print('Helpers defined.')

In [ ]:
def get_storm_data(key, cfg):
    date_str = cfg['date']
    print(f'\n{key}  ({date_str})')
    rec = {
        'date':         date_str[:10],
        'precip_type':  cfg['precip_type'],
        'avg_temp_f':   None,
        'soil_wetness': None,
        'peak_wind_mph': None,
    }
    try:
        H = Herbie(date_str, model='hrrr', product='sfc', fxx=0, verbose=False)

        # 10-m wind speed (m/s -> mph)
        try:
            ds_w = H.xarray('WIND:10 m above ground')
            wind_mph = extract_to_grid(ds_w) * 2.23694
            rec['peak_wind_mph'] = np.round(wind_mph, 1).tolist()
            avg_w  = np.nanmean(wind_mph)
            peak_w = np.nanmax(wind_mph)
            print(f'  wind  avg={avg_w:.1f}  peak={peak_w:.1f} mph')
        except Exception as e:
            print(f'  wind  FAILED: {e}')

        # 2-m air temperature (K -> degF)
        try:
            ds_t = H.xarray('TMP:2 m above ground')
            tmp_f = (extract_to_grid(ds_t) - 273.15) * 9 / 5 + 32
            tf = round(float(np.nanmean(tmp_f)), 1)
            rec['avg_temp_f'] = tf
            print(f'  temp  avg={tf} F')
        except Exception as e:
            print(f'  temp  FAILED: {e}')

        # Soil moisture 0-10 cm (GRIB label varies by HRRR version)
        for pat in ('SOILW:0-0.1 m below ground level', 'SOILW:0-0.1 m', 'SOILW'):
            try:
                ds_s = H.xarray(pat)
                sw = round(float(np.nanmean(extract_to_grid(ds_s))), 3)
                rec['soil_wetness'] = sw
                print(f'  soil  avg={sw} m3/m3  (label: {pat!r})')
                break
            except Exception:
                pass
        else:
            print('  soil  FAILED (no SOILW pattern matched)')

    except Exception as e:
        print(f'  HRRR FAILED: {e}')
    return rec


print('Downloading HRRR data (~2-5 min per storm from AWS archive)...')
storm_results = {}
for key, cfg in STORMS.items():
    storm_results[key] = get_storm_data(key, cfg)
print('\nAll downloads complete.')

In [ ]:
output = {
    '_populated': True,
    'grid': {
        'lats':  TARGET_LATS.tolist(),
        'lons':  TARGET_LONS.tolist(),
        'n_lat': int(len(TARGET_LATS)),
        'n_lon': int(len(TARGET_LONS)),
        'note':  'peak_wind_mph is row-major [n_lat][n_lon]. Bilinear-interpolate to (lat,lon).',
    },
    'storms': {
        k: {'name': STORM_NAMES[k], **storm_results[k]}
        for k in STORMS
    },
}

header = (
    '// HRRR wind speed, temperature, soil moisture \u2014 Hartford County storms.\n'
    '// Generated by fetch_hrrr_storm_wind.ipynb \u2014 do not hand-edit.\n'
    '// Source: NOAA HRRR via AWS archive (herbie-data library).\n\n'
)
body = 'window.HARTFORD_STORM_WIND = ' + json.dumps(output, indent=2) + ';\n'

with open('hartford_storm_wind.js', 'w') as f:
    f.write(header + body)

n_ok = sum(1 for s in output['storms'].values() if s['peak_wind_mph'] is not None)
ns   = len(output['storms'])
print(f'Wrote hartford_storm_wind.js  ({n_ok}/{ns} storms with wind data)')
print('Download via Files panel \u2192 copy to EnerygyOptimization/data/')

In [ ]:
print('Storm summary:')
for key, s in output['storms'].items():
    wind = s.get('peak_wind_mph')
    tf   = s.get('avg_temp_f')
    sw   = s.get('soil_wetness')
    if wind:
        peak = max(max(row) for row in wind)
        avg  = np.nanmean(wind)
        print(f'  {key:<15}  wind avg={avg:.0f} peak={peak:.0f} mph  temp={tf} F  soil={sw}')
    else:
        print(f'  {key:<15}  wind=MISSING  temp={tf} F  soil={sw}')